In [8]:
import gzip
import sys
from pathlib import Path

import warnings
import itertools

import numpy as np
import pandas as pd

import scipy
from scipy import stats


import matplotlib as mpl
import matplotlib.pyplot as plt

from matplotlib.ticker import FuncFormatter
from matplotlib.gridspec import GridSpec
from matplotlib.colors import LinearSegmentedColormap

import seaborn as sns

from IPython.display import display
warnings.filterwarnings("ignore")

import korflab

In [9]:
# ============================================
# Global plotting configuration
# ============================================

_PLOT_CFG = {
    "fig_w": 6.0,
    "fig_h": 5.0,
    "dpi": 300,
}


def set_plot_style(
    *,
    # Fonts
    base_fontsize=11,
    title_fontsize=14,
    label_fontsize=13,
    tick_fontsize=11,
    legend_fontsize=11,

    dpi=300,

    axes_linewidth=1.2,
    spines_top=True,
    spines_right=True,

    tick_size_major=6,
    tick_dir="out",

    grid=False,

    fig_w=6.0,
    fig_h=5.0,
):

    sns.set_style("ticks")

    mpl.rcParams.update({

        # =====================================
        # Fonts
        # =====================================
        "font.family": "DejaVu Sans",
        "font.size": base_fontsize,

        "axes.titlesize": title_fontsize,
        "axes.labelsize": label_fontsize,

        "xtick.labelsize": tick_fontsize,
        "ytick.labelsize": tick_fontsize,

        "legend.fontsize": legend_fontsize,

        # =====================================
        # Figure
        # =====================================
        "figure.dpi": dpi,
        "savefig.dpi": dpi,

        # =====================================
        # Axes
        # =====================================
        "axes.linewidth": axes_linewidth,
        "axes.spines.top": spines_top,
        "axes.spines.right": spines_right,
        "axes.grid": grid,
        "axes.axisbelow": True,

        # =====================================
        # Ticks
        # =====================================
        "xtick.major.size": tick_size_major,
        "ytick.major.size": tick_size_major,
        "xtick.direction": tick_dir,
        "ytick.direction": tick_dir,

        # =====================================
        # Legend
        # =====================================
        "legend.frameon": False,

        # =====================================
        # Save figure
        # =====================================
        "savefig.bbox": "tight",
        "savefig.transparent": False,
    })

    _PLOT_CFG.update({
        "fig_w": fig_w,
        "fig_h": fig_h,
        "dpi": dpi,
    })


def make_fig(
    w=None,
    h=None,
    dpi=None,
):
    W = float(w) if w is not None else _PLOT_CFG["fig_w"]
    H = float(h) if h is not None else _PLOT_CFG["fig_h"]
    D = dpi if dpi is not None else _PLOT_CFG["dpi"]

    fig, ax = plt.subplots(
        figsize=(W, H),
        dpi=D,
    )

    return fig, ax


set_plot_style()

In [10]:
# ============================================
# Input FASTQ files
# ============================================

fastq_dir = Path("/mnt/h/1k_fastq")

fastq_files = sorted(fastq_dir.glob("*_1k.fastq.gz"))

for f in fastq_files:
    print(f.name)

CR.log.P_1k.fastq.gz
CR.log.T_1k.fastq.gz
CR.stat.P_1k.fastq.gz
CR.stat.T_1k.fastq.gz
sample1_1k.fastq.gz
sample3_1k.fastq.gz
sample5_1k.fastq.gz
sample6_1k.fastq.gz


In [11]:
# ============================================
# Sample metadata
# ============================================

records = []

for f in fastq_files:
    name = f.name.replace("_1k.fastq.gz", "")

    if name.startswith("sample"):
        dataset = "New"
    else:
        dataset = "Old"

    records.append({
        "Sample": name,
        "Dataset": dataset,
        "Path": f,
    })

sample_meta = pd.DataFrame(records)

display(sample_meta)

,Sample,Dataset,Path
0,CR.log.P,Old,/mnt/h/1k_fastq/CR.log.P_1k.fastq.gz
1,CR.log.T,Old,/mnt/h/1k_fastq/CR.log.T_1k.fastq.gz
2,CR.stat.P,Old,/mnt/h/1k_fastq/CR.stat.P_1k.fastq.gz
3,CR.stat.T,Old,/mnt/h/1k_fastq/CR.stat.T_1k.fastq.gz
4,sample1,New,/mnt/h/1k_fastq/sample1_1k.fastq.gz
5,sample3,New,/mnt/h/1k_fastq/sample3_1k.fastq.gz
6,sample5,New,/mnt/h/1k_fastq/sample5_1k.fastq.gz
7,sample6,New,/mnt/h/1k_fastq/sample6_1k.fastq.gz


In [12]:
# ============================================
# Read one FASTQ
# ============================================

def summarize_fastq(path, sample, dataset):

    records = []

    for header, seq, plus, qual in korflab.readfastq(path):

        length = len(seq)

        q = np.fromiter(
            (ord(c) - 33 for c in qual),
            dtype=float,
        )

        mean_q = q.mean()

        gc = (
            (seq.count("G") + seq.count("C"))
            / length
            * 100
        )

        records.append({
            "Dataset": dataset,
            "Sample": sample,
            "Read_ID": header,
            "Length": length,
            "Mean_Q": mean_q,
            "GC": gc,
        })

    return pd.DataFrame(records)